# AI-Assisted CAD Design — Physics M³

Interactive notebook for parametric design with AI guidance.
**Pipeline:** Material → CAD → Mesh → FEM → KDI M³

In [ ]:
import sys, os, math, json
import numpy as np
import importlib.util as _ilu

_PROJ = "/home/cnmfs/bioeolica-dev2/workspaces"

def _imp(rel, name):
    s = _ilu.spec_from_file_location(name, os.path.join(_PROJ, rel))
    m = _ilu.module_from_spec(s)
    sys.modules.setdefault(name, m)
    s.loader.exec_module(m)
    return m

print("Módulos carregados")
print(f"  CadQuery: {_imp('cad-cae-platform/modules/cad_bridge.py', 'cb')}")
print(f"  Gmsh:     {_imp('cad-cae-platform/modules/gmsh_mesher.py', 'gm')}")
print(f"  CalculiX: {_imp('cad-cae-platform/modules/calculix_solver.py', 'cx')}")
print(f"  Material: Importando physics-m3...")

In [ ]:
sys.path.insert(0, os.path.join(_PROJ, 'physics-m3'))
from modules.composite_model import CompositeMaterial
from modules.structural_analysis import von_mises_stress, safety_factor, tsai_wu_failure
from modules.mechanical_tests import tensile_test
print("Módulos de engenharia carregados")

## 1. Material Compósito
Configure fiber, matrix, coating, volume fraction → E1, E2, G12, nu12

In [ ]:
fiber = 'waste_paper'
matrix = 'pva'
coating = 'graphite_coating'
V_f = 0.25

mat = CompositeMaterial(fiber=fiber, matrix=matrix, coating=coating)
ec = mat.elastic_constants()
print(f"Material: {fiber}/{matrix}/{coating} Vf={V_f}")
print(f"  E1 = {ec['E1_longitudinal_GPa']:.1f} GPa")
print(f"  E2 = {ec['E2_transverse_GPa']:.1f} GPa")
print(f"  G12 = {ec['G12_shear_GPa']:.2f} GPa")

## 2. CAD Design
Parâmetros → geometria 3D → STEP export

In [ ]:
cb = _imp('cad-cae-platform/modules/cad_bridge.py', 'cb')

L, W, H = 120, 20, 8  # mm
viga = cb.cantilever_beam(length=L, width=W, height=H)
print(f"Viga: {L}x{W}x{H} mm")
print(f"  Volume: {viga.volume:.0f} mm³")
print(f"  Massa:  {viga.mass:.4f} kg")

# Export STEP
step_path = "/tmp/viga_design.step"
viga.export_step(step_path)
print(f"\nSTEP exportado: {step_path}")

## 3. KDI M³ Analysis
Macro (ambiente) → Meso (concentração) → Micro (homogeneização)

In [ ]:
km = _imp('kdi-m3-bridge/modules/kdi_macro.py', 'km')
kme = _imp('kdi-m3-bridge/modules/kdi_meso.py', 'kme')
kmi = _imp('kdi-m3-bridge/modules/kdi_micro.py', 'kmi')

env = km.MacroEnvironment(altitude_m=100, wind_speed_ref_ms=30)
ma = km.MacroAnalysis(cad_model=viga, env=env)
r = ma.run()
print(f"Macro:  Volume={r['volume_mm3']:.0f} mm³")
print(f"        Vento={r['environment']['wind_pressure_kPa']:.2f} kPa")

rm = kme.MesoAnalysis({"nominal_stress_MPa": 50, "yield_strength_MPa": 250}).run()
print(f"Meso:   Kt={rm['Kt']}  SF={rm['safety_factor']}")

rmi = kmi.MicroAnalysis(fiber=fiber, matrix=matrix, coating=coating, V_f=V_f).run()
print(f"Micro:  E1={rmi['E1_GPa']} GPa")
print(f"        Material: {rmi['material']}")

## 4. Análise de Falha
von Mises + Safety Factor + Tsai-Wu

In [ ]:
vm = von_mises_stress(50, 10, 5)
sf = safety_factor(250, vm)
tw = tsai_wu_failure(50, 10, 5, Xt=200, Xc=150, Yt=50, Yc=100, S=30)
print(f"von Mises: {vm:.1f} MPa")
print(f"Safety Factor: {sf:.2f}")
print(f"Tsai-Wu: {tw}")

## 5. Otimização DOE
Design of Experiments + Pareto

In [ ]:
dq = _imp('cad-cae-platform/modules/design_optimizer.py', 'dq')
ds = dq.DesignSpace({"L": (60, 200), "W": (10, 30)})
opt = dq.DesignOptimizer(ds, ["mass", "stiffness"])
results = opt.run_doe(lambda p: {"mass": p["L"]*p["W"]*8/1000, "stiffness": 1/(p["L"]*p["W"]*8)})
pareto = opt.pareto_frontier()
print(f"DOE: {len(results)} designs, {len(pareto)} Pareto-ótimos")

---
**Physics M³ — CAD/CAE Platform**

Pipeline: Material → CAD → Macro → Meso → Micro → Falha → DOE